In [4]:
%load_ext Cython

In [5]:
%%cython

# cython: language_level=3
from libc.stdint cimport uint32_t

cdef uint32_t modular_exponentiation(uint32_t base, int exp, uint32_t modulus):
    cdef uint32_t result = 1
    cdef uint32_t current_base = base % modulus

    while exp > 0:
        if exp % 2 == 1:
            result = (result * current_base) % modulus
        current_base = (current_base * current_base) % modulus
        exp //= 2

    return result

cpdef list rabin_karp_search_with_boundaries(str text, list patterns, int a, uint32_t m):
    # Convert the entire text to lowercase
    text = text.lower()
    cdef int N = len(text)
    cdef list results = []
    cdef str pattern
    cdef int M, i
    cdef uint32_t pattern_hash, text_hash

    for pattern in patterns:
        # Convert the pattern to lowercase
        pattern = pattern.lower()
        M = len(pattern)
        inverse_power = modular_exponentiation(a, M-1, m)
        pattern_hash = rabin_hash(pattern, a, m)
        text_hash = rabin_hash(text[:M], a, m)

        for i in range(N - M + 1):

            # brute_force_hash = rabin_hash(text[i:(i+M)], a, m)
            # if text_hash != brute_force_hash:
            #     print('mismatch at', i, text_hash, '!=', brute_force_hash)
            
            if text_hash == pattern_hash and (
                    (i == 0 or not text[i-1].isalnum()) and
                    (i + M == N or not text[i+M].isalnum())
                ):
                if text[i:i+M] == pattern:
                    results.append((i, pattern))
            
            if i < N - M:
                text_hash = (text_hash - ord(text[i]) * inverse_power) % m
                text_hash = (text_hash * a + ord(text[i + M])) % m

    return results

cdef uint32_t rabin_hash(str s, int a, uint32_t m):
    cdef uint32_t hash_val = 0
    for ch in s:
        hash_val = (hash_val * a + ord(ch)) % m
    return hash_val


Content of stdout:
_cython_magic_d5262e756aada13d673f5b4eb62c9650ed55f080.c
C:\Users\olooney\.ipython\cython\_cython_magic_d5262e756aada13d673f5b4eb62c9650ed55f080.c(2527): warning C4244: '=': conversion from 'Py_ssize_t' to 'int', possible loss of data
C:\Users\olooney\.ipython\cython\_cython_magic_d5262e756aada13d673f5b4eb62c9650ed55f080.c(2597): warning C4244: '=': conversion from 'Py_ssize_t' to 'int', possible loss of data
   Creating library C:\Users\olooney\.ipython\cython\Users\olooney\.ipython\cython\_cython_magic_d5262e756aada13d673f5b4eb62c9650ed55f080.cp311-win_amd64.lib and object C:\Users\olooney\.ipython\cython\Users\olooney\.ipython\cython\_cython_magic_d5262e756aada13d673f5b4eb62c9650ed55f080.cp311-win_amd64.exp
Generating code
Finished generating code

In [6]:
text = "Hello, how are you today? I am fine. However, I am antsy. How's that? It's a beautiful day."
rabin_karp_search_with_boundaries(text, patterns=["how", "Day", "am", "beautiful"], a=131, m=10007)

[(7, 'how'),
 (58, 'how'),
 (87, 'day'),
 (28, 'am'),
 (48, 'am'),
 (77, 'beautiful')]

In [8]:
with open(r'D:/Dropbox/books/war_and_peace.txt') as fin:
    text = fin.read()

In [11]:
rabin_karp_search_with_boundaries(text, patterns=["Godfreys", "0068V00000yF5ImQAK"], a=131, m=10007)

[(3101286, 'godfreys'),
 (3101862, 'godfreys'),
 (3101964, 'godfreys'),
 (3102022, 'godfreys')]

In [12]:
%%timeit
rabin_karp_search_with_boundaries(text, patterns=["Godfreys"], a=131, m=10007)

337 ms ± 22.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
%%timeit
text.index('006PB000006XORWYA4')

In [13]:
import re
patterns = ["how", "Day", "am", "beautiful"]
text = "Hello, how are you today? I am fine. However, I am antsy. How's that? It's a beautiful day."

def find_all(text, patterns):
    regex = re.compile(r'\b(' + "|".join(patterns) + r')\b', re.IGNORECASE)
    for match in regex.finditer(text):
        yield match.start(), match.group()
    
list(find_all(text, patterns))

[(7, 'how'),
 (28, 'am'),
 (48, 'am'),
 (58, 'How'),
 (77, 'beautiful'),
 (87, 'day')]

In [14]:
with open(r'D:/Dropbox/books/war_and_peace.txt') as fin:
    text = fin.read()

In [15]:
list(find_all(text, patterns=["Godfreys", "0068V00000yF5ImQAK"]))

[(3101286, 'Godfreys'),
 (3101862, 'Godfreys'),
 (3101964, 'Godfreys'),
 (3102022, 'Godfreys')]

In [16]:
%%timeit
list(find_all(text, patterns=["Godfreys", "0068V00000yF5ImQAK"]))

89 ms ± 2.13 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [17]:
%%timeit
list(find_all(text, patterns=[
    "Oncology", "Cancer", "Tumor", "Carcinoma", "Malignant",
    "Benign", "Metastasis", "Chemotherapy", "Radiation",
    "Oncologist", "Neoplasm", "Biopsy", "Lymphoma", "Leukemia",
    "Melanoma", "Carcinogen", "Mastectomy", "Prostate", "Radiology",
    "Immunotherapy", "Pathology", "Hematology", "Sarcoma",
    "Glioma", "Myeloma", "Oncogene", "Antineoplastic", "Brachytherapy",
    "Adenocarcinoma"
]))


474 ms ± 18.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
from rk.rabin_karp import rabin_karp_search_with_boundaries

In [ ]:
rabin_karp_search_with_boundaries(text, patterns=["Canada", "0068V00000yF5ImQAK"], a=131, m=10007)

In [ ]:
%%timeit
rabin_karp_search_with_boundaries(text, patterns=["0068V00000yF5ImQAK"], a=131, m=10007)